# Chapter 04: BM25 Ranking with Terms

Exact declared variant: k1=1.2, b=0.75. Worked cache calculation and controls.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root by walking parents from the current working directory.
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

# Make this notebook's shared helper importable, then re-resolve through it.
NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)

## Worked calculation for the command passage

In [ ]:
from lexical_index import bm25_term

# Frozen baseline: N=13 eligible rows, average length 9 tokens.
n = 13
average = 9
length = 7

cache_contribution = bm25_term(f=2, df=3, n=n, length=length, average=average, k1=1.2, b=0.75)
reset_contribution = bm25_term(f=1, df=5, n=n, length=length, average=average, k1=1.2, b=0.75)
total = cache_contribution + reset_contribution

print("cache contribution:", repr(cache_contribution))
print("reset contribution:", repr(reset_contribution))
print("command passage total:", repr(total))
assert abs(total - 3.060972) < 1e-5

## Saturation: repeated terms flatten

In [ ]:
for frequency in range(1, 8):
    value = bm25_term(f=frequency, df=3, n=13, length=7, average=9, k1=1.2, b=0.75)
    print(f"f={frequency}: {value:.6f}")

## Length sensitivity

In [ ]:
for ratio in (0.5, 1.0, 1.5):
    passage_length = int(9 * ratio)
    value = bm25_term(f=2, df=3, n=13, length=passage_length, average=9, k1=1.2, b=0.75)
    print(f"length/average={ratio}: {value:.6f}")

## Ranking result on the cache query

In [ ]:
from document_identity import Collection
from lexical_index import InvertedIndex

collection = Collection.load(ROOT / "examples" / "fixtures" / "part1-v1.json")
index = InvertedIndex.from_collection(collection, "baseline")
bm25_result = index.search("cache reset", method="bm25", limit=3)
print([(r["passage_id"], round(r["score"], 6)) for r in bm25_result["retrieved"]])
assert bm25_result["selected"][0]["score"] > bm25_result["selected"][1]["score"]
assert abs(bm25_result["selected"][0]["score"] - 3.060972) < 1e-5

## Controls: overlap versus BM25

In [ ]:
overlap_result = index.search("cache reset", method="overlap", limit=3)
print("overlap:", [(r["passage_id"], r["score"]) for r in overlap_result["retrieved"]])
print("bm25:  ", [(r["passage_id"], round(r["score"], 6)) for r in bm25_result["retrieved"]])

## Invalid parameters are rejected

In [ ]:
def expect_value_error(func):
    try:
        func()
    except ValueError:
        return True
    raise AssertionError("expected ValueError")

assert expect_value_error(lambda: bm25_term(1, 0, 13, 7, 9, 1.2, 0.75))
assert expect_value_error(lambda: bm25_term(1, 3, 13, 7, 9, -1.0, 0.75))
assert expect_value_error(lambda: bm25_term(1, 3, 13, 7, 9, 1.2, 1.5))

## Optional plot (graceful fallback)

In [ ]:
try:
    import matplotlib.pyplot as plt
    xs = list(range(1, 8))
    ys = [bm25_term(f=f, df=3, n=13, length=7, average=9, k1=1.2, b=0.75) for f in xs]
    plt.plot(xs, ys, marker="o")
    plt.xlabel("term frequency")
    plt.ylabel("contribution")
    plt.title("BM25 saturation at k1=1.2")
    plt.show()
except Exception as exc:
    print("matplotlib unavailable or skipped:", type(exc).__name__)

## Try it yourself
Vary k1 and b, or inspect `reset` postings. Remember the score belongs to the indexed eligible view.

In [ ]:
print(index.inspect("reset")["df"])

## Observed versus not established
Observed: arithmetic, saturation, length sensitivity and one ranking change. Not established: general superiority, truth, permission or downstream usefulness.